# Integrated Gradients Visualization

Map per-atom IG attribution scores onto structure files as B-factors,
then save as mmCIF files for visualization in interactive Mol\* widget, or locally in PyMOL / ChimeraX.

Supports two structure sources:
1. **Graph archive** (`.pngrph`): Reconstructs atom arrays from the pre-built archive used during training or inference.
2. **Raw CIF file**: Parses the original structure file directly.

Optionally appends IG summary stats (output path, mean score, max score) to an existing prediction result file.

In [ ]:
from pathlib import Path

import biotite.structure as bs
import biotite.structure.io.pdbx as pdbx
import numpy as np
import pandas as pd
import atomworks.constants as awconst

from protnote.utils.graph_archive import open_archive
from protnote.utils.structure import parse_structure, extract_aa_residue_by_chain_ids

# Reverse mapping: atomic number -> element symbol (for archive reconstruction)
_ATOMIC_NUMBER_TO_ELEMENT = {v: k for k, v in awconst.ELEMENT_NAME_TO_ATOMIC_NUMBER.items()}
_ATOMIC_NUMBER_TO_ELEMENT[0] = "X"  # unknown

## Configuration

In [ ]:
# ── Structure source ──
# Graph directory: auto-detects single graphs.pngrph or sharded graphs.shard-*-of-*.pngrph
# Set to None to fall back to raw CIF files.
graph_dir = Path("../data/processed")

# Fallback: raw CIF files directory (only used when no archive is found in graph_dir)
cif_dir = Path("../data/structures/pdb")
chain_ids = ["A"] # e.g. ["A"] # Chain filter

# ── IG scores ──
# Directory of .npy files named {seq_id}.npy, each of shape [N_atoms]
# TODO: change score loading respecting to actual interpretability implementation
ig_scores_dir = Path("../outputs/ig_scores")

# ── Output ──
output_dir = Path("../outputs/ig_bfactor_cifs")

# ── Sequence IDs to process (None = all available in IG scores dir) ──
sequence_ids = None  # or e.g. ["P12345", "Q67890"]

# ── Optional: prediction result file to append IG stats ──
# Set to None to skip result file update.
# Supports .h5 (HDF5) and .parquet formats (same formats saved by bin/main.py).
result_file_path = None  # e.g. Path("../outputs/results/test_1_logits_my_run.h5")

## Helper functions

In [ ]:
def graph_dict_to_atom_array(graph_data: dict) -> bs.AtomArray:
    """Reconstruct a Biotite AtomArray from a graph archive entry.

    The archive stores coords, atom_types (atomic numbers), atom_names,
    residue_index, residue_names, and residue_res_ids — enough to build
    a valid AtomArray for mmCIF export.
    """
    coords = graph_data["coords"].numpy().astype(np.float64)
    atom_types = graph_data["atom_types"].numpy()
    atom_names = graph_data["atom_names"]
    residue_index = graph_data["residue_index"].numpy()
    residue_names = graph_data["residue_names"]
    residue_res_ids = graph_data["residue_res_ids"].numpy()
    n_atoms = len(coords)

    aa = bs.AtomArray(n_atoms)
    aa.coord = coords
    aa.atom_name = np.array(atom_names)
    aa.element = np.array([_ATOMIC_NUMBER_TO_ELEMENT.get(int(at), "X") for at in atom_types])
    aa.res_name = np.array([residue_names[residue_index[i]] for i in range(n_atoms)])
    aa.res_id = np.array([int(residue_res_ids[residue_index[i]]) for i in range(n_atoms)])
    aa.chain_id = np.array(["A"] * n_atoms)
    aa.set_annotation("occupancy", np.ones(n_atoms))
    return aa


def load_atom_array(seq_id: str, archive_reader, cif_dir, chain_ids):
    """Load atom array from archive (preferred) or raw CIF file."""
    if archive_reader is not None and seq_id in archive_reader:
        graph_data = archive_reader[seq_id]
        return graph_dict_to_atom_array(graph_data)
    # Fallback: raw CIF
    cif_path = cif_dir / f"{seq_id}.cif"
    if not cif_path.exists():
        # Try lowercase
        cif_path = cif_dir / f"{seq_id.lower()}.cif"
    if not cif_path.exists():
        raise FileNotFoundError(f"No structure found for {seq_id} in archive or {cif_dir}")
    atom_array = parse_structure(cif_path)
    return extract_aa_residue_by_chain_ids(atom_array, chain_ids)


def write_ig_cif(atom_array: bs.AtomArray, ig_scores: np.ndarray, output_path: Path):
    """Set IG scores as B-factors and write mmCIF."""
    atom_array.set_annotation("b_factor", ig_scores.astype(np.float64))
    if not hasattr(atom_array, "occupancy"):
        atom_array.set_annotation("occupancy", np.ones(atom_array.array_length()))
    cif_file = pdbx.CIFFile()
    pdbx.set_structure(cif_file, atom_array)
    cif_file.write(str(output_path))

## Initialize structure reader and discover sequence IDs

In [ ]:
# Detect archive: single .pngrph file or sharded directory (same logic as bin/main.py)
archive_reader = None
if graph_dir is not None:
    graph_dir = Path(graph_dir)
    default_archive = graph_dir / "graphs.pngrph"
    if default_archive.exists():
        archive_reader = open_archive(default_archive)
        print(f"Archive reader (single file): {len(archive_reader)} entries from {default_archive}")
    elif any(graph_dir.glob("graphs.shard-*-of-*.pngrph")):
        archive_reader = open_archive(graph_dir)
        print(f"Archive reader (sharded): {len(archive_reader)} entries from {graph_dir}")
    else:
        print(f"No .pngrph archive found in {graph_dir}, falling back to raw CIF files from {cif_dir}")

if archive_reader is None:
    print(f"Using raw CIF files from {cif_dir}")

# Discover sequence IDs from IG scores directory
if sequence_ids is None:
    ig_files = sorted(Path(ig_scores_dir).glob("*.npy"))
    sequence_ids = [f.stem for f in ig_files]
    print(f"Discovered {len(sequence_ids)} sequence IDs from {ig_scores_dir}")
else:
    print(f"Processing {len(sequence_ids)} user-specified sequence IDs")

output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

## Process all structures

In [ ]:
ig_records = []  # (seq_id, output_path, mean_ig, max_ig)
errors = []

for seq_id in sequence_ids:
    ig_path = ig_scores_dir / f"{seq_id}.npy"
    if not ig_path.exists():
        errors.append((seq_id, "IG scores file not found"))
        continue

    ig_scores = np.load(ig_path)

    try:
        atom_array = load_atom_array(seq_id, archive_reader, cif_dir, chain_ids)
    except (FileNotFoundError, KeyError) as e:
        errors.append((seq_id, str(e)))
        continue

    n_atoms = atom_array.array_length()
    if ig_scores.shape[0] != n_atoms:
        errors.append((seq_id, f"IG size {ig_scores.shape[0]} != structure atoms {n_atoms}"))
        continue

    out_path = output_dir / f"{seq_id}_ig.cif"
    write_ig_cif(atom_array, ig_scores, out_path)

    mean_ig = float(ig_scores.mean())
    max_ig = float(ig_scores.max())
    ig_records.append((seq_id, str(out_path.resolve()), mean_ig, max_ig))
    print(f"  {seq_id}: {n_atoms} atoms | mean IG={mean_ig:.6f} | max IG={max_ig:.6f} -> {out_path.name}")

print(f"\nProcessed: {len(ig_records)} / {len(sequence_ids)}")
if errors:
    print(f"Errors ({len(errors)}):")
    for sid, msg in errors:
        print(f"  {sid}: {msg}")

## (Optional) Append IG stats to prediction result file

If `result_file_path` is set, this adds three columns to the existing logits DataFrame:
- `ig_cif_path`: absolute path to the output mmCIF with IG B-factors
- `ig_mean`: mean IG score across all atoms
- `ig_max`: max IG score across all atoms

In [ ]:
if result_file_path is not None and len(ig_records) > 0:
    result_file_path = Path(result_file_path)
    suffix = result_file_path.suffix.lower()

    # Read existing result file
    if suffix == ".h5":
        # Try common HDF5 keys used by save_evaluation_results
        for key in ["logits_df", "labels_df", "df"]:
            try:
                result_df = pd.read_hdf(result_file_path, key=key)
                hdf_key = key
                break
            except KeyError:
                continue
        else:
            raise KeyError(f"No recognized key found in {result_file_path}")
    elif suffix == ".parquet":
        result_df = pd.read_parquet(result_file_path)
    else:
        raise ValueError(f"Unsupported result file format: {suffix}. Use .h5 or .parquet.")

    print(f"Loaded result file: {result_file_path} ({len(result_df)} rows)")

    # Build IG stats DataFrame indexed by sequence_id
    ig_df = pd.DataFrame(ig_records, columns=["sequence_id", "ig_cif_path", "ig_mean", "ig_max"])
    ig_df = ig_df.set_index("sequence_id")

    # Drop existing IG columns if re-running
    for col in ["ig_cif_path", "ig_mean", "ig_max"]:
        if col in result_df.columns:
            result_df = result_df.drop(columns=[col])

    # Join on index (sequence_id)
    result_df = result_df.join(ig_df, how="left")

    n_matched = result_df["ig_mean"].notna().sum()
    print(f"Matched {n_matched}/{len(ig_records)} IG records to result file rows")

    # Save back in the same format
    if suffix == ".h5":
        result_df.to_hdf(result_file_path, key=hdf_key, mode="w")
    else:
        result_df.to_parquet(result_file_path)

    print(f"Updated: {result_file_path}")
    print(result_df[["ig_cif_path", "ig_mean", "ig_max"]].dropna().head())
else:
    if result_file_path is None:
        print("Skipping result file update (result_file_path is None)")
    else:
        print("Skipping result file update (no records to write)")

## Interactive 3D Visualization (ipymolstar)

This section is **self-contained** — it reads directly from `output_dir` (the directory of `*_ig.cif` files), so you can skip the processing cells above if structures were already exported in a previous run.

Set `vis_seq_id` below to pick a specific structure, or leave as `None` to randomly select one from the output directory.

In [ ]:
# Show the color palette for Integrated Gradient scores

from IPython.display import HTML, display

_legend_html = """
<table style="border-collapse:collapse; font-family:sans-serif; font-size:14px;">
  <tr style="border-bottom:2px solid #ccc;">
    <th style="padding:6px 12px; text-align:left;">Category</th>
    <th style="padding:6px 12px; text-align:left;">IG Range</th>
    <th style="padding:6px 12px; text-align:left;">Color</th>
  </tr>
  <tr>
    <td style="padding:6px 12px;">Very high</td>
    <td style="padding:6px 12px;">0.9 – 1.0</td>
    <td style="padding:6px 12px;"><span style="display:inline-block;width:60px;height:18px;background:#106DFF;border:1px solid #888;border-radius:3px;"></span></td>
  </tr>
  <tr>
    <td style="padding:6px 12px;">Confident</td>
    <td style="padding:6px 12px;">0.7 – 0.9</td>
    <td style="padding:6px 12px;"><span style="display:inline-block;width:60px;height:18px;background:#10CFF1;border:1px solid #888;border-radius:3px;"></span></td>
  </tr>
  <tr>
    <td style="padding:6px 12px;">Low</td>
    <td style="padding:6px 12px;">0.5 – 0.7</td>
    <td style="padding:6px 12px;"><span style="display:inline-block;width:60px;height:18px;background:#F6ED12;border:1px solid #888;border-radius:3px;"></span></td>
  </tr>
  <tr>
    <td style="padding:6px 12px;">Very low</td>
    <td style="padding:6px 12px;">0.0 – 0.5</td>
    <td style="padding:6px 12px;"><span style="display:inline-block;width:60px;height:18px;background:#EF821E;border:1px solid #888;border-radius:3px;"></span></td>
  </tr>
</table>
"""
display(HTML(_legend_html))

In [ ]:
import random
from ipymolstar import PDBeMolstar

# IG attribution color palette (adapted from AlphaFold pLDDT scheme)
IG_COLOR_LUT = {
    "very-high": {"r": 16, "g": 109, "b": 255},   # #106DFF blue
    "confident": {"r": 16, "g": 207, "b": 241},    # #10CFF1 cyan
    "low":       {"r": 246, "g": 237, "b": 18},     # #F6ED12 yellow
    "very-low":  {"r": 239, "g": 130, "b": 30},     # #EF821E orange
}

def ig_category(value: float) -> str:
    if value >= 0.9:
        return "very-high"
    elif value >= 0.7:
        return "confident"
    elif value >= 0.5:
        return "low"
    else:
        return "very-low"


def build_ig_color_and_tooltip_data(cif_path: Path):
    """Read the IG-annotated mmCIF and build per-residue color_data and tooltips.

    Residue-level IG is the mean B-factor (= IG score) across atoms in that residue.
    """
    cif_file = pdbx.CIFFile.read(str(cif_path))
    atoms = pdbx.get_structure(cif_file, model=1, extra_fields=["b_factor"])

    # Aggregate per-residue mean IG from atom-level B-factors
    residue_starts = []
    prev_key = None
    for i in range(atoms.array_length()):
        key = (atoms.chain_id[i], int(atoms.res_id[i]))
        if key != prev_key:
            residue_starts.append(i)
            prev_key = key
    residue_starts.append(atoms.array_length())

    color_query = []
    tooltip_query = []
    for idx in range(len(residue_starts) - 1):
        start = residue_starts[idx]
        end = residue_starts[idx + 1]
        chain = atoms.chain_id[start]
        resn = int(atoms.res_id[start])
        mean_ig = float(atoms.b_factor[start:end].mean())
        cat = ig_category(mean_ig)

        color_query.append({
            "struct_asym_id": chain,
            "residue_number": resn,
            "color": IG_COLOR_LUT[cat],
        })
        tooltip_query.append({
            "struct_asym_id": chain,
            "residue_number": resn,
            "tooltip": f"IG: {cat} ({mean_ig:.4f})",
        })

    color_data = {"data": color_query, "nonSelectedColor": None}
    tooltip_data = {"data": tooltip_query}
    return color_data, tooltip_data

In [ ]:
# Pick structure to visualize
vis_seq_id = None  # Set to a specific seq_id, or None to pick randomly
vis_output_dir = Path("../outputs/ig_bfactor_cifs")  # Directory containing *_ig.cif files

# Discover available IG-annotated structures from the output directory
available_cifs = sorted(vis_output_dir.glob("*_ig.cif"))
available_ids = {p.name.removesuffix("_ig.cif"): p for p in available_cifs}

if not available_ids:
    raise FileNotFoundError(f"No *_ig.cif files found in {vis_output_dir}. Run the processing cells first.")
else:
    print(f"Found {len(available_ids)} IG structures in {vis_output_dir}")

    if vis_seq_id is None:
        vis_seq_id = random.choice(list(available_ids.keys()))
        print(f"Randomly selected: {vis_seq_id}")
    elif vis_seq_id not in available_ids:
        raise ValueError(f"{vis_seq_id} not found. Available: {list(available_ids.keys())}")

    vis_cif_path = available_ids[vis_seq_id]
    print(f"Visualizing: {vis_seq_id}  ({vis_cif_path})")

    # Load CIF content as string for inline display
    cif_content = vis_cif_path.read_text()
    custom_data = {"data": cif_content, "format": "cif", "binary": False}

    # Build per-residue color and tooltip data from IG B-factors
    color_data, tooltip_data = build_ig_color_and_tooltip_data(vis_cif_path)

    view = PDBeMolstar(
        custom_data=custom_data,
        color_data=color_data,
        tooltips=tooltip_data,
        hide_water=True,
        hide_carbs=True,
        theme="light",
    )

In [ ]:
# Show the visualization widget
view